# 11-bob. Si asosida autentifikatsiya va firibgarlikni aniqlash
## Kengaytirilgan amaliy ish: o'tkazma hisoblari tarmog'ini topish

[![Colab'da ochish](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrmsi/si-kiberxavfsizlik/blob/main/11-bob/11-3_otkazma-hisoblari-tarmogini-topish.ipynb)

*"Sun'iy intellektni kiberxavfsizlikda qo'llash" darsligi, 11-bob.*

> Bu kengaytirilgan amaliy ish darslikning **60610700 – Sun'iy intellekt yo'nalishi versiyasida** keltirilgan.

Quyidagi dasturda 2000 ta oddiy mijoz va 15 ta o'tkazma hisobidan iborat graf yaratiladi. Mijozlarning bir qismi oilaviy qurilmani bo'lishadi va uy internet manzillari takrorlanadi, 60 ta mijoz esa tadbirkor bo'lib, ular ko'p kiruvchi o'tkazma oladi. O'tkazma hisoblari uchta qurilma va ikkita IP manzildan foydalanadi, ayrimlari esa boshqa "toza" qurilmadan ham kiradi. Bank 15 ta o'tkazma hisobidan faqat 3 tasini biladi. Ikki usul taqqoslanadi: faqat kiruvchi o'tkazmalar soni bo'yicha saralash va ma'lum firibgarlardan boshlanuvchi shaxsiylashtirilgan PageRank.

**Ishga tushirish:** yuqoridagi menyudan *Runtime → Run all* (*Ishga tushirish → Hammasini bajarish*) ni tanlang yoki kataklarni birma-bir Shift+Enter bilan bajaring. Barcha ma'lumotlar dasturning o'zida yaratiladi, qo'shimcha fayl yuklash shart emas.

In [ ]:
import networkx as nx
import numpy as np

rng = np.random.default_rng(11)
G = nx.Graph()

In [ ]:
# 1. 2000 ta oddiy mijoz: har birining o'z qurilmasi, ba'zilari oilaviy qurilmani bo'lishadi
mijozlar = [f"m{i}" for i in range(2000)]
for i, m in enumerate(mijozlar):
    G.add_edge(m, f"qurilma{i // 2 if i < 400 else i}")          # 400 mijoz juft bo'lib bitta qurilma
    G.add_edge(m, f"ip{rng.integers(0, 600)}")                   # uy internet manzillari takrorlanadi

In [ ]:
# 2. Firibgarlar tarmog'i: 15 ta "o'tkazma" hisobi, 3 ta qurilma va 2 ta IP manzilda
mulelar = [f"mule{i}" for i in range(15)]
for i, m in enumerate(mulelar):
    G.add_edge(m, f"f_qurilma{i % 3}")
    G.add_edge(m, f"f_ip{i % 2}")
    if rng.random() < 0.3:                                        # ayrimlari "toza" qurilmadan ham kiradi
        G.add_edge(m, f"qurilma{rng.integers(1000, 2000)}")

In [4]:
# 3. Tranzaksiya faolligi: kiruvchi o'tkazmalar soni (oddiy mijozlarda ham katta bo'lishi mumkin)
kiruvchi = {m: int(rng.poisson(3)) for m in mijozlar}
for m in rng.choice(mijozlar, 60, replace=False):
    kiruvchi[m] += int(rng.integers(15, 40))                      # tadbirkorlar, savdo hisoblari
kiruvchi.update({m: int(rng.integers(8, 20)) for m in mulelar})
hisoblar = mijozlar + mulelar

# Bank 3 ta mule hisobini allaqachon aniqlagan; qolgan 12 tasini topish kerak
malum = mulelar[:3]
qidiriladi = set(mulelar[3:])

def eng_yuqori(ball, k=20):
    nomzodlar = [h for h in hisoblar if h not in malum]
    return sorted(nomzodlar, key=lambda h: -ball[h])[:k]

# A usul: faqat tranzaksiya faolligi
topA = eng_yuqori(kiruvchi)
# B usul: ma'lum firibgarlardan boshlanuvchi shaxsiylashtirilgan PageRank (xavfning graf bo'ylab tarqalishi)
pr = nx.pagerank(G, alpha=0.85, personalization={h: (1 if h in malum else 0) for h in G})
topB = eng_yuqori(pr)

print(f"Graf: {G.number_of_nodes()} tugun, {G.number_of_edges()} qirra")
print(f"A usul (kiruvchi o'tkazmalar soni): top-20 da {len(qidiriladi & set(topA))}/12 mule")
print(f"B usul (graf bo'ylab xavf tarqalishi): top-20 da {len(qidiriladi & set(topB))}/12 mule")
print("B usul bo'yicha birinchi 15 ta hisob:", ", ".join(topB[:15]))

Graf: 4403 tugun, 4033 qirra
A usul (kiruvchi o'tkazmalar soni): top-20 da 0/12 mule
B usul (graf bo'ylab xavf tarqalishi): top-20 da 12/12 mule
B usul bo'yicha birinchi 15 ta hisob: mule12, mule8, mule14, mule6, mule5, mule4, mule10, mule11, mule3, mule9, mule7, mule13, m1772, m1594, m1058


## Kitobdagi natija

Dasturni ishga tushirganingizda quyidagiga o'xshash natija chiqishi kerak (tasodifiy sonlar generatori urug'i qat'iy belgilangani uchun natija odatda aynan bir xil bo'ladi; faqat bajarilish vaqti va har safar yangi yaratiladigan tasodifiy kalitlar farq qilishi mumkin):

```
Graf: 4403 tugun, 4033 qirra
A usul (kiruvchi o'tkazmalar soni): top-20 da 0/12 mule
B usul (graf bo'ylab xavf tarqalishi): top-20 da 12/12 mule
B usul bo'yicha birinchi 15 ta hisob: mule12, mule8, mule14, mule6, mule5, mule4, mule10, mule11, mule3, mule9, mule7, mule13, m1772, m1594, m1058
```

## O'zingiz sinab ko'ring

Parametrlarni (ma'lumotlar hajmi, chegara qiymati, tasodifiy son urug'i) o'zgartirib, natija qanday o'zgarishini kuzating. Kitobdagi 11-bobning B darajadagi topshiriqlari aynan shu daftar asosida bajariladi.